# EON Cloud ML Runner (Colab)

Sovereign worker template for **EON sovereign AI**. This notebook runs **in the cloud (Colab GPU)** and executes ML jobs handed to it by the local thin client. The local machine never imports torch/tensorflow — all compute happens here.

- Gateway: `GATEWAY_URL` (env) or `http://127.0.0.1:8787`
- Job source: `JOB_URL` (env) or inline `JOB_JSON` (env)
- Mirror: `http://<MIRROR_HOST>/models/<version>/...` or rclone `mirror:models/<version>`


## 1. Fetch the job
Pull a job JSON from the webhook URL / env `JOB_URL`, or read the inline `JOB_JSON` env var.

In [ ]:
import json, os, urllib.request

GATEWAY_URL = os.environ.get("GATEWAY_URL", "http://127.0.0.1:8787")
MIRROR_HOST = os.environ.get("MIRROR_HOST", "127.0.0.1")
JOB_URL = os.environ.get("JOB_URL", "") or f"{GATEWAY_URL}/api/ml/job/latest"

def _fetch(url):
    with urllib.request.urlopen(url, timeout=120) as r:
        return r.read().decode("utf-8")

if os.environ.get("JOB_JSON"):
    job = json.loads(os.environ["JOB_JSON"])
else:
    job = json.loads(_fetch(JOB_URL))

TASK_ID = job.get("task_id")
CODE = job.get("code", "result = 42")
DATA = job.get("data", {})
FRAMEWORK = job.get("framework", "torch").lower()
VERSION = str(job.get("version", "latest"))
print("task_id =", TASK_ID, "| framework =", FRAMEWORK, "| version =", VERSION)


## 2. Install the framework
`torch` by default; `tensorflow` when `framework == "tf"`.

In [ ]:
if FRAMEWORK == "tf":
    %pip install --quiet tensorflow
else:
    %pip install --quiet torch


## 3. Download data / weights from the sovereign mirror
Fetch `job["files"]` from `http://<MIRROR_HOST>/models/<version>/...`, or fall back to rclone from MEGA.

In [ ]:
import os, shutil, subprocess, urllib.request

os.makedirs("weights", exist_ok=True)
for name in job.get("files", []):
    dst = os.path.join("weights", name)
    url = f"http://{MIRROR_HOST}/models/{VERSION}/{name}"
    try:
        urllib.request.urlretrieve(url, dst)
        print("mirror download ok:", name)
    except Exception as e:
        print(f"mirror miss ({e}); trying rclone")
        if shutil.which("rclone"):
            subprocess.run(["rclone", "copy", f"mirror:models/{VERSION}", "weights"], check=False)


## 4. Execute the job code
Run the provided `code` string with `exec(code, {"data": data})`. Output is captured to `result["stdout"]`.

In [ ]:
import contextlib, io

buf = io.StringIO()
ok, error = True, None
try:
    with contextlib.redirect_stdout(buf):
        exec(compile(CODE, "<colab-job>", "exec"), {"data": DATA, "task_id": TASK_ID})
except Exception as e:
    ok, error = False, repr(e)
result = {"stdout": buf.getvalue(), "error": error}
print(json.dumps(result))


## 5. Upload results & notify the gateway
Save `result.json` under `/content/<version>`, upload to `/mnt/fluid-cloud/models/<version>/` via rclone (or HTTP POST to the mirror), then POST `{task_id, status, result, provider: "colab"}` back to `GATEWAY_URL/api/ml/complete`.

In [ ]:
import json, os, shutil, subprocess, urllib.request

out_dir = f"/content/{VERSION}"
os.makedirs(out_dir, exist_ok=True)
with open(os.path.join(out_dir, "result.json"), "w") as f:
    json.dump(result, f, indent=2)

if shutil.which("rclone"):
    rc = subprocess.run(["rclone", "copy", out_dir, f"mirror:models/{VERSION}"]).returncode
    print("rclone upload rc =", rc)

payload = {
    "task_id": TASK_ID,
    "status": "done" if ok else "failed",
    "result": result,
    "provider": "colab",
}
req = urllib.request.Request(
    f"{GATEWAY_URL}/api/ml/complete",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=120) as r:
    print("webhook:", r.read().decode("utf-8"))

print("EON Colab runner done. status =", payload["status"])
